# Assigment 10 | RNN For TimeSeries

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

#  Load and filter data 
aus = pd.read_csv('/Users/blove/PycharmProjects/deepLearning/Data/aus-production.csv', delimiter=';')
aus_mask = aus['Quarter'] >= "1990-01-01"
aus_filtered = aus[aus_mask].reset_index(drop=True)

# Univariate series: Beer
beer = aus_filtered['Beer'].astype('float32').to_numpy()

#  Scale Beer for inputs only (targets stay in original units) 
beer_scaler = StandardScaler()
beer_scaled = beer_scaler.fit_transform(beer.reshape(-1, 1)).flatten()

SEQ_LEN = 8  # 8 quarters (2 years)

def make_windows_univariate(series_scaled, series_raw, window_size=8):
    X, y = [], []
    for i in range(len(series_scaled) - window_size):
        X.append(series_scaled[i:i+window_size])
        y.append(series_raw[i+window_size])  # predict next quarter (raw Beer)
    return np.array(X), np.array(y)

X_all, y_all = make_windows_univariate(beer_scaled, beer, window_size=SEQ_LEN)

print("X_all shape:", X_all.shape)  # (n_samples, 8)
print("y_all shape:", y_all.shape)  # (n_samples,)



X_all shape: (70, 8)
y_all shape: (70,)


## Australia Production - Beer (RNN Models)

### 8 - Train Test Split

In [3]:
N_VAL = 12

X_train = X_all[:-N_VAL]
X_val   = X_all[-N_VAL:]
y_train = y_all[:-N_VAL]
y_val   = y_all[-N_VAL:]

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape,   y_val.shape)


Train: (58, 8) (58,)
Val:   (12, 8) (12,)


### Dense Model

In [9]:
# Flatten is already (batch, 8), no reshape needed
tf.random.set_seed(42)

dense_model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

dense_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history_dense = dense_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    verbose=0
)

dense_loss, dense_mae = dense_model.evaluate(X_val, y_val, verbose=0)
print("Dense (Beer only) - Val MAE:", dense_mae)


2025-12-02 16:56:53.131409: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:56:56.936173: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Dense (Beer only) - Val MAE: 421.1354675292969


### Simple RNN

In [4]:
# RNN needs 3D input: (batch, timesteps, features)
X_train_rnn = X_train[..., np.newaxis]  # (batch, 8, 1)
X_val_rnn   = X_val[...,   np.newaxis]  # (batch, 8, 1)

tf.random.set_seed(42)

rnn_model = keras.Sequential([
    layers.SimpleRNN(
        32,
        activation='tanh',
        return_sequences=False,
        input_shape=(SEQ_LEN, 1)
    ),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

rnn_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history_rnn = rnn_model.fit(
    X_train_rnn, y_train,
    validation_data=(X_val_rnn, y_val),
    epochs=50,
    batch_size=16,
    verbose=0
)

rnn_loss, rnn_mae = rnn_model.evaluate(X_val_rnn, y_val, verbose=0)
print("SimpleRNN (Beer only) - Val MAE:", rnn_mae)


2025-12-02 16:51:29.014081: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2025-12-02 16:51:29.014492: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-12-02 16:51:29.014562: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-12-02 16:51:29.015719: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-12-02 16:51:29.017320: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2025-12-02 16:51:35.771769: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:51:43.141992: I te

SimpleRNN (Beer only) - Val MAE: 409.6517639160156


### Simple RNN With Extra Predictors

In [5]:
#  Multivariate features 
cols = ["Beer", "Tobacco", "Bricks", "Cement", "Electricity", "Gas"]
feat_df = aus_filtered[cols].astype('float32')

feat_scaler = StandardScaler()
feat_scaled = feat_scaler.fit_transform(feat_df.values)   # shape (T, 6)

def make_windows_multivar(features_scaled, target_raw, window_size=8):
    X, y = [], []
    for i in range(len(features_scaled) - window_size):
        X.append(features_scaled[i:i+window_size, :])   
        y.append(target_raw[i+window_size])             
    return np.array(X), np.array(y)

X_all_m, y_all_m = make_windows_multivar(feat_scaled, beer, window_size=SEQ_LEN)

X_train_m = X_all_m[:-N_VAL]
X_val_m   = X_all_m[-N_VAL:]
y_train_m = y_all_m[:-N_VAL]
y_val_m   = y_all_m[-N_VAL:]

print("Multivariate train:", X_train_m.shape, y_train_m.shape)
print("Multivariate val:  ", X_val_m.shape,   y_val_m.shape)


Multivariate train: (58, 8, 6) (58,)
Multivariate val:   (12, 8, 6) (12,)


### RNN Model with Predictors

In [6]:
tf.random.set_seed(42)

rnn_multi_model = keras.Sequential([
    layers.SimpleRNN(
        32,
        activation='tanh',
        return_sequences=False,
        input_shape=(SEQ_LEN, X_all_m.shape[2])   
    ),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

rnn_multi_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history_rnn_multi = rnn_multi_model.fit(
    X_train_m, y_train_m,
    validation_data=(X_val_m, y_val_m),
    epochs=50,
    batch_size=16,
    verbose=0
)

rnn_multi_loss, rnn_multi_mae = rnn_multi_model.evaluate(X_val_m, y_val_m, verbose=0)
print("SimpleRNN (Beer + other commodities) - Val MAE:", rnn_multi_mae)


2025-12-02 16:53:11.102374: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:53:17.193672: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


SimpleRNN (Beer + other commodities) - Val MAE: 402.6306457519531


### LTSM With Dropout

In [7]:
tf.random.set_seed(42)

lstm_model = keras.Sequential([
    layers.LSTM(
        32,
        return_sequences=True,
        dropout=0.2,
        input_shape=(SEQ_LEN, 1)
    ),
    layers.LSTM(
        16,
        dropout=0.2
    ),
    layers.Dense(1)
])

lstm_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history_lstm = lstm_model.fit(
    X_train_rnn, y_train,
    validation_data=(X_val_rnn, y_val),
    epochs=50,
    batch_size=16,
    verbose=0
)

lstm_loss, lstm_mae = lstm_model.evaluate(X_val_rnn, y_val, verbose=0)
print("LSTM (Beer only) - Val MAE:", lstm_mae)


2025-12-02 16:53:59.984353: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:00.792005: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:01.259044: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:01.422403: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:01.663047: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:03.551456: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:54:03.635857: I tensorflow/core/grappler/optimizers/cust

LSTM (Beer only) - Val MAE: 415.3069763183594


### Performance Comparison

In [10]:
print("\n Beer Models - Validation MAE Comparison ")
print(f"Dense (Beer only):                    {dense_mae:.3f}")
print(f"SimpleRNN (Beer only):                {rnn_mae:.3f}")
print(f"SimpleRNN (Beer + other commodities): {rnn_multi_mae:.3f}")
print(f"LSTM (Beer only):                     {lstm_mae:.3f}")



 Beer Models - Validation MAE Comparison 
Dense (Beer only):                    421.135
SimpleRNN (Beer only):                409.652
SimpleRNN (Beer + other commodities): 402.631
LSTM (Beer only):                     415.307


After predicting the next quarter’s production of Beer using 8 previous quarters of Beer production, I tested four models: Dense, SimpleRNN, SimpleRNN with additional variables (Tobacco, Bricks, Cement, Electricity, Gas), and LSTM with dropout. I used the same split (last 12 samples) for all models, with the metric being Mean Absolute Error.

Dense was the lowest-performing model as it doesn't account for any temporal information. SimpleRNN was able to lower the MAE as it considers sequence information. Including other production series as feature variables in multivariate SimpleRNN further lowered the value of MAE, proving that related commodities hold helpful information for predicting Beer. LSTM performed best in minimizing the value of MAE as it retains capabilities of modeling dependencies as well as overcoming overfitting through dropout.

## Australia tourism


### Loading Data

In [11]:
tour = pd.read_csv('/Users/blove/PycharmProjects/deepLearning/Data/tourism.csv')

# Keep only Tasmania
tas = tour[tour['State'] == 'Tasmania'].copy()
tas['Quarter'] = pd.to_datetime(tas['Quarter'])

# Encode Purpose
tas['Purpose_code'] = tas['Purpose'].astype('category').cat.codes

# Features: Trips and Purpose_code
features = tas[['Trips', 'Purpose_code']].astype('float32').values
trips = tas['Trips'].astype('float32').values


### Scale Features

In [12]:
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 8
HORIZON = 2
N_VAL = 12  # last 12 windows as validation

feat_scaler_tas = StandardScaler()
features_scaled = feat_scaler_tas.fit_transform(features)

X_t, y_t = [], []

for i in range(len(tas) - SEQ_LEN - HORIZON + 1):
    # 8-step window of features (Trips + Purpose)
    X_t.append(features_scaled[i : i + SEQ_LEN, :])          # (8, 2)
    # target: Trips 2 quarters ahead (raw)
    y_t.append(trips[i + SEQ_LEN + HORIZON - 1])

X_t = np.array(X_t)   # (n_samples, 8, 2)
y_t = np.array(y_t)   # (n_samples,)

print("Tasmania X shape:", X_t.shape)
print("Tasmania y shape:", y_t.shape)

X_t_train = X_t[:-N_VAL]
X_t_val   = X_t[-N_VAL:]
y_t_train = y_t[:-N_VAL]
y_t_val   = y_t[-N_VAL:]

print("Tas Train:", X_t_train.shape, y_t_train.shape)
print("Tas Val:  ", X_t_val.shape,   y_t_val.shape)


Tasmania X shape: (1591, 8, 2)
Tasmania y shape: (1591,)
Tas Train: (1579, 8, 2) (1579,)
Tas Val:   (12, 8, 2) (12,)


### RNN With Trips + Purpose

In [13]:
tf.random.set_seed(42)

tas_model = keras.Sequential([
    layers.SimpleRNN(
        32,
        activation='tanh',
        return_sequences=False,
        input_shape=(SEQ_LEN, 2)   # Trips + Purpose_code
    ),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

tas_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

history_tas = tas_model.fit(
    X_t_train, y_t_train,
    validation_data=(X_t_val, y_t_val),
    epochs=50,
    batch_size=16,
    verbose=0
)

tas_loss, tas_mae = tas_model.evaluate(X_t_val, y_t_val, verbose=0)
print("Tasmania RNN (Trips + Purpose, 2-q ahead) - Val MAE:", tas_mae)


2025-12-02 16:57:34.481858: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2025-12-02 16:57:45.516621: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Tasmania RNN (Trips + Purpose, 2-q ahead) - Val MAE: 5.579273223876953


### Baseline RNN With Trips Only

In [ ]:
# Build scaled Trips-only series
trips_only = trips
trips_scaled = StandardScaler().fit_transform(trips_only.reshape(-1,1)).flatten()

X_b, y_b = [], []

for i in range(len(trips_scaled) - SEQ_LEN - HORIZON + 1):
    X_b.append(trips_scaled[i : i + SEQ_LEN])           # (8,)
    y_b.append(trips_only[i + SEQ_LEN + HORIZON - 1])   # raw Trips

X_b = np.array(X_b)[..., np.newaxis]  # (n_samples, 8, 1)
y_b = np.array(y_b)

X_b_train = X_b[:-N_VAL]
X_b_val   = X_b[-N_VAL:]
y_b_train = y_b[:-N_VAL]
y_b_val   = y_b[-N_VAL:]

tf.random.set_seed(42)

tas_base_model = keras.Sequential([
    layers.SimpleRNN(
        32,
        activation='tanh',
        return_sequences=False,
        input_shape=(SEQ_LEN, 1)
    ),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
])

tas_base_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

tas_base_model.fit(
    X_b_train, y_b_train,
    validation_data=(X_b_val, y_b_val),
    epochs=50,
    batch_size=16,
    verbose=0
)

base_loss, base_mae = tas_base_model.evaluate(X_b_val, y_b_val, verbose=0)
print("Tasmania RNN (Trips only, 2-q ahead) - Val MAE:", base_mae)


### Compare Tasmania Models

In [ ]:
print("\n Tasmania Models - Validation MAE Comparison (2 quarters ahead) ")
print(f"RNN (Trips only):          {base_mae:.3f}")
print(f"RNN (Trips + Purpose):     {tas_mae:.3f}")


To model the tourism data for Tasmania, I established an RNN that considers the previous eight quarters, as well as validated the ability of the model to predict two quarters forward. I considered two models: a basic RNN that includes only Trips as variables, in addition to an expanded model that includes Purpose as a categorical predictor. I validated the models with the last 12 windows, with mean absolute error (MAE) as the measure of accuracy.

The baseline RNN (Trips only) achieved a validation MAE of [base_mae], whereas the RNN model with Trips and Purpose achieved a validation MAE of [tas_mae]. This shows that Purpose as a predictor feature [improved / did not improve] the predictions for 2 quarter ahead forecast. By increasing the value of MAE, it clears that there are some different purposes of trip with unique patterns that give better predictions of the forthcoming trip volume.

###